## Image exploration
### Activate caiman environment with conda-forge (mamba)

In [6]:
!conda init bash

# !conda activate caiman
!pip list | grep caiman

no change     /home/abl-dell/miniforge3/condabin/conda
no change     /home/abl-dell/miniforge3/bin/conda
no change     /home/abl-dell/miniforge3/bin/activate
no change     /home/abl-dell/miniforge3/bin/deactivate
no change     /home/abl-dell/miniforge3/etc/profile.d/conda.sh
no change     /home/abl-dell/miniforge3/etc/fish/conf.d/conda.fish
no change     /home/abl-dell/miniforge3/shell/condabin/Conda.psm1
no change     /home/abl-dell/miniforge3/shell/condabin/conda-hook.ps1
no change     /home/abl-dell/miniforge3/lib/python3.13/site-packages/xontrib/conda.xsh
no change     /home/abl-dell/miniforge3/etc/profile.d/conda.csh
no change     /home/abl-dell/.bashrc
No action taken.
caiman                        1.13.1


### Check structure and shape of files

In [7]:
import h5py
import numpy as np

path = r"/home/abl-dell/Downloads/caiman_dataset/2026-04-20_144321-20260614T141833Z-3-002/2026-04-20_144321/raw/stack_1-A1-GCaMP8M_channel_1_obj_bottom/Cam_long_00000.lux.h5"

def explore_h5(filepath):
    with h5py.File(filepath, "r") as f:

        # ---- top-level structure ----
        print("=== Top-level keys ===")
        f.visititems(lambda name, obj: print(f"  {name}  [{type(obj).__name__}]  "
                                            f"shape={getattr(obj, 'shape', '—')}  "
                                            f"dtype={getattr(obj, 'dtype', '—')}"))

        # ---- root-level attributes (microscope metadata) ----
        print("\n=== Root attributes ===")
        for k, v in f.attrs.items():
            print(f"  {k}: {v}")

        # ---- Data array specifics ----
        if "Data" in f:
            d = f["Data"]
            shape = d.shape
            print(f"\n=== Data array ===")
            print(f"  shape  : {shape}")
            print(f"  dtype  : {d.dtype}")
            print(f"  ndim   : {d.ndim}")

            if d.ndim == 3:
                T, H, W = shape
                print(f"  → interpreted as  T={T}  H={H}  W={W}")
                print(f"  → if this is a z-stack movie: {T} time-points or {T} Z-planes")
            elif d.ndim == 4:
                T, Z, H, W = shape
                print(f"  → interpreted as  T={T}  Z={Z}  H={H}  W={W}")
                print(f"  → {T} time-points  ×  {Z} Z-planes")

            # Data attributes (per-dataset metadata)
            print("\n=== Data attributes ===")
            for k, v in d.attrs.items():
                print(f"  {k}: {v}")

            # Quick intensity sanity check (reads only first frame)
            first = d[0].astype(np.float32)
            print(f"\n=== First frame stats ===")
            print(f"  min={first.min():.1f}  max={first.max():.1f}  "
                f"mean={first.mean():.1f}  nonzero={np.count_nonzero(first)}")
            
explore_h5(path)

=== Top-level keys ===
  Data  [Dataset]  shape=(4900, 2048, 2048)  dtype=uint16
  metadata  [Dataset]  shape=()  dtype=object

=== Root attributes ===

=== Data array ===
  shape  : (4900, 2048, 2048)
  dtype  : uint16
  ndim   : 3
  → interpreted as  T=4900  H=2048  W=2048
  → if this is a z-stack movie: 4900 time-points or 4900 Z-planes

=== Data attributes ===
  element_size_um: [0.0033 0.208  0.208 ]

=== First frame stats ===
  min=155.0  max=54321.0  mean=7359.2  nonzero=4194304


In [8]:
with h5py.File(path, "r") as f:
      import json
      raw = f["metadata"][()]
      meta = json.loads(raw)
      print(json.dumps(meta, indent=2))

{
  "processingInformation": {
    "version": "1.0.0",
    "image_id": "2026-04-20T18:43:26.789Z-b6207737-524b-468c-ab5f-beb7492ab8d5",
    "sources": [
      "Luxendo TruLive3D, Embedded v3.17.3, serial-nr: 40073"
    ],
    "contains_beads": false,
    "time_point": "0",
    "channel": "1",
    "stack": "1",
    "stack_description": "A1-GCaMP8M",
    "objective": "bottom",
    "camera": "long",
    "stack_scan_ids": [
      "oc:default_st:1_ch:1_tp:0_pm:none"
    ],
    "voxel_size_um": {
      "width": 0.208,
      "height": 0.208,
      "depth": 0.0033068121
    },
    "image_size_vx": {
      "width": 2048,
      "height": 2048,
      "depth": 4900
    },
    "affine_to_sample": [
      {
        "matrix": [
          [
            1.0,
            0.0,
            0.0
          ],
          [
            0.0,
            1.0,
            0.0
          ],
          [
            0.0,
            0.0,
            1.0
          ]
        ],
        "translation": [
          -1023.5

In [9]:
with h5py.File(path, "r") as f:
    d = f["Data"]          # shape (4900, 2048, 2048)
    n_planes = 7
    plane_idx = 3          # middle plane (0–6), change as needed

    # 700 time points for a single z-plane
    movie = d[plane_idx::n_planes]   # shape (700, 2048, 2048)
    print(movie.shape)               # → (700, 2048, 2048)

(700, 2048, 2048)
